---
title: "Filter, Group, Aggregate"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- Boolean masks: the filter dropdown and `COUNTIFS`, written down
- `groupby`: the PivotTable and the `COUNTIFS` table at once
- `pivot_table` and row shares: the two-way table from chapter 2
- `merge`: the `XLOOKUP` join from chapter 3, with the misses counted
- The twelve checks and the `keep` column, in pandas
- An Excel-to-pandas dictionary for everything in chapters 1 to 3

**The one idea**

> Three verbs cover everything the Excel chapters did. A filter is a column of `True`/`False` used to pick rows. A pivot is a `groupby`. A lookup is a `merge`. Once you can say which of the three a question needs, the pandas line is short, and the Excel version is the check.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_05_lecture.ipynb){download="chapter_05_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_05/chapter_05_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_05/`.
:::

## Back to the decision

Chapters 2 and 3 told the board that Metro has two populations of riders, that weekends are the busiest days per day, that Monthly Pass holders prefer electric bikes, that the Westside out-rides downtown per station, and that the headline average was inflated by undocked bikes. Chapter 4 rebuilt chapter 1 in pandas. This chapter rebuilds the rest, and the test of every line is the same: it must give the number the workbook gave.

By the end, the entire Excel analysis is one notebook. That is the handoff the title promises, and it is also the point at which Python starts to pay for itself: the next quarter's file runs through the whole thing in under a minute.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

LOCAL = "../../data/raw/"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/")
src = LOCAL if os.path.exists(LOCAL + "metro_trips.csv") else URL

trips = pd.read_csv(src + "metro_trips.csv")
stations = pd.read_csv(src + "metro_stations.csv", encoding="cp1252").dropna(how="all")
stations.columns = [c.strip() for c in stations.columns]
stations["Kiosk ID"] = stations["Kiosk ID"].astype(int)

trips["start_time"] = pd.to_datetime(trips["start_time"])
trips["hour"] = trips["start_time"].dt.hour                      # Excel: =HOUR(C2)
trips["weekday"] = trips["start_time"].dt.strftime("%a")         # Excel: =TEXT(C2, "ddd")
trips.shape, stations.shape

The two helper columns from chapter 2, `hour` and `weekday`, are the last two lines. In Excel each was a formula filled down 88,932 rows; here each is one line, and `dt` is the door to everything a date-time knows (`.dt.date`, `.dt.month`, `.dt.day_name()`).

## Filter

Chapter 4 showed `d > 120`: a comparison on a column gives a column of `True`/`False`, called a mask. Put the mask inside the square brackets and pandas keeps the rows where it is `True`. That is the filter dropdown.

In [ ]:
walk = trips[trips["passholder_type"] == "Walk-up"]
len(walk)

Two conditions join with `&` (and) or `|` (or), each in its own parentheses. This is `COUNTIFS` with two pairs:

In [ ]:
# Excel: =COUNTIFS(O:O, "electric", B:B, ">60")
mask = (trips["bike_type"] == "electric") & (trips["duration"] > 60)
int(mask.sum())

The parentheses are not optional: without them, Python reads `"electric" & trips["duration"]` first and stops with an error. The rule is one condition per pair of parentheses, always.

A mask is reusable. Name it once and use it in a count, a filter, and an average without retyping the condition:

In [ ]:
long_electric = trips[mask]
print(round(long_electric["duration"].mean(), 1), "minutes on average")
long_electric["passholder_type"].value_counts().head(2)

## Group

`groupby` is the PivotTable and the `COUNTIFS` twin at once. Name the column to group by, then say what to compute. `size()` counts rows; `mean()` averages; `agg` does several at once.

In [ ]:
# Excel: the by weekday sheet (chapter 2), counts and average minutes
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_wd = (trips.groupby("weekday")
              .agg(trips=("trip_id", "size"), avg_minutes=("duration", "mean"))
              .reindex(days))
by_wd.round(1)

Check it against the workbook. The counts match the `by weekday` sheet exactly (Saturday 13,969). The `reindex(days)` line is the fix for pivots sorting weekdays alphabetically; here it is one word instead of dragging row labels. And `agg(trips=("trip_id", "size"))` reads as "a column called `trips`, made by taking `trip_id` and counting": name first, then the pair of column and function.

Chapter 2's second idea was *per what?* The quarter had 13 of some weekdays and 12 of others. In pandas the denominator is one more `groupby`, on the calendar rather than on the trips:

In [ ]:
cal = pd.Series(pd.date_range("2026-01-01", "2026-03-31").strftime("%a"))
by_wd["days"] = cal.value_counts().reindex(days)
by_wd["per_day"] = by_wd["trips"] / by_wd["days"]
by_wd.round(1)

Wednesday's 1,010 per day and Saturday's 1,075 are the numbers from the chapter 2 table. The division is a column divided by a column, row by row, which is what filling a formula down a column does.

The by-hour table is the same pattern with `hour`, and the chart is one line more:

In [ ]:
#| label: fig-by-hour
#| fig-cap: "Trips by hour of day from a one-line groupby. Same shape as the chapter 2 line, two peaks at 8 and 17."
by_hour = trips.groupby("hour").size()
ax = by_hour.plot(figsize=(9, 3.6), color="#1F4E79", linewidth=2)
ax.set_xticks(range(0, 24, 4)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Trips in the quarter")
ax.set_ylim(0, by_hour.max() * 1.1)
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

## Two-way tables

The `two ways` sheet put pass type in rows and bike type in columns. That is `pivot_table`, and the arguments are the four boxes of Excel's field list: `index` is Rows, `columns` is Columns, `values` and `aggfunc` are Values.

In [ ]:
# Excel: the two ways sheet, counts
two_way = pd.pivot_table(trips, index="passholder_type", columns="bike_type",
                         values="trip_id", aggfunc="size")
two_way

Row shares, which chapter 2 argued were the only fair comparison between groups of different sizes, are a division of each row by its total. `div` with `axis=0` means "divide each row by the matching entry of this Series":

In [ ]:
shares = two_way.div(two_way.sum(axis=1), axis=0)
shares.round(3)

Monthly Pass 56.3% electric, Walk-up 24.1%: the chapter 2 table, to the decimal. If it had not matched, the first suspect would be the pivot's source range in Excel, not the pandas line; a filtered or partial range is the usual way a pivot goes wrong, and `pivot_table` has no such failure mode because it always sees the whole frame.

## Join

Chapter 3's `XLOOKUP` pulled a station name and region onto every trip by its id. In pandas that is `merge`, and the arguments map onto the four questions from the `lookup` sheet: `left_on` is "what am I looking for", `right_on` is "where do I look", the right frame's other columns are "what do I bring back", and `how="left"` is "keep every trip, even the ones with no match".

In [ ]:
lookup = (stations[["Kiosk ID", "Kiosk Name", "Region"]]
          .rename(columns={"Kiosk ID": "start_station", "Kiosk Name": "start_name", "Region": "start_region"}))
joined = trips.merge(lookup, on="start_station", how="left", indicator=True)
joined["_merge"].value_counts()

`indicator=True` adds a column saying whether each row matched. `both` is a match, `left_only` is the `"not found"` of chapter 3, and there are none this quarter, which is what the `checks` sheet found. The row count is unchanged, 88,932, which is the thing to verify after every left join: if it grew, the lookup table had duplicate ids and every duplicated trip is now counted twice.

What chapter 3 found by hand shows up here as a blank: station 3000 is in the table with no region, so its trips joined to nothing.

In [ ]:
joined["start_region"] = joined["start_region"].fillna("(no region)")
# Excel: the by region sheet
by_region = (joined.groupby("start_region")
                   .agg(trips=("trip_id", "size"), avg_minutes=("duration", "mean"),
                        stations_used=("start_station", "nunique"))
                   .sort_values("trips", ascending=False))
by_region["share"] = by_region["trips"] / len(joined)
by_region["per_station"] = by_region["trips"] / by_region["stations_used"]
by_region.round({"avg_minutes": 1, "share": 3, "per_station": 0})

DTLA 60.0% of trips across 151 stations, Westside 516 trips per station against DTLA's 353: the chapter 3 numbers. `nunique` is `ROWS(UNIQUE(...))`, counting the distinct stations that saw a trip, and it took the `SUMPRODUCT` acrobatics of the Excel version down to one word.

## Clean

The twelve checks from chapter 3 are each a mask summed. Here are the ones that found something, with the Excel formula beside each:

In [ ]:
d = trips["duration"]
same = trips["start_station"] == trips["end_station"]
mins = (pd.to_datetime(trips["end_time"]) - trips["start_time"]).dt.total_seconds() / 60
checks = pd.Series({
    "duplicate ids   =ROWS(A:A)-ROWS(UNIQUE(A:A))":  trips["trip_id"].duplicated().sum(),
    "blank station   =COUNTBLANK(E:E)":              trips["start_station"].isna().sum(),
    "1 minute or less  =COUNTIFS(B:B,\"<=1\")":      (d <= 1).sum(),
    "at the cap   =COUNTIFS(B:B,\">=1440\")":        (d >= 1440).sum(),
    "round trip <= 2 min   =SUMPRODUCT((E=H)*(B<=2))": (same & (d <= 2)).sum(),
    "duration vs timestamps   =SUMPRODUCT(--(ABS(..)>2))": ((mins - d).abs() > 2).sum(),
})
checks

3,211, 287, 3,239, 298: the `checks` sheet, line for line. The `keep` column from the chapter 3 exercise is one mask, and before-and-after is the same aggregate on two frames:

In [ ]:
keep = (d > 1) & (d < 1440)
before_after = pd.DataFrame({
    "all rows": [d.mean(), d.median(), d[trips["passholder_type"] == "Walk-up"].mean()],
    "kept rows": [d[keep].mean(), d[keep].median(), d[keep & (trips["passholder_type"] == "Walk-up")].mean()],
}, index=["average minutes", "median minutes", "walk-up average"])
before_after.round(1)

32.9 to 29.3, the median unmoved, the walk-up average down eleven minutes. Same table as the chapter 3 solution workbook.

One habit worth building now: write the mask, name it, and apply it with `.loc` when you want both a row filter and a column pick in one step. `trips.loc[keep, "duration"]` reads as "the duration of the kept rows". It is the form you will see in every pandas codebase, and it avoids a warning pandas raises when you filter and then assign.

## The dictionary

Everything in chapters 1 to 3, with its pandas equivalent. This table is the handoff; keep it open until the right column is the one you reach for first.

| Excel | pandas |
|---|---|
| Open the CSV | `pd.read_csv(path)` |
| Scroll to the bottom; count columns | `df.shape` |
| The data dictionary sheet | `df.dtypes` |
| `=COUNTA(B:B)`, `=AVERAGE(B:B)`, `=MEDIAN(B:B)` | `d.count()`, `d.mean()`, `d.median()` |
| `=COUNTIF(B:B, ">120")` | `(d > 120).sum()` |
| `=COUNTIFS(O:O, "electric", B:B, ">60")` | `((df.bike_type == "electric") & (df.duration > 60)).sum()` |
| Filter dropdown | `df[mask]` or `df.loc[mask, cols]` |
| Helper column `=HOUR(C2)` | `df["start_time"].dt.hour` |
| PivotTable, one field in Rows | `df.groupby(col).size()` or `.agg(...)` |
| PivotTable, Rows and Columns | `pd.pivot_table(df, index=, columns=, values=, aggfunc=)` |
| Row shares | `tab.div(tab.sum(axis=1), axis=0)` |
| `=XLOOKUP(E2, ids, names, "not found")` | `df.merge(lookup, on=, how="left")` |
| `=ROWS(UNIQUE(A:A))` | `df["trip_id"].nunique()` |
| `=COUNTBLANK(E:E)` | `df["start_station"].isna().sum()` |
| Remove Duplicates | `df.drop_duplicates("trip_id")` |
| Sort largest first | `df.sort_values(col, ascending=False)` |
| A line chart of a column | `series.plot()` |

## Back to the decision

Nothing new for the board. Every finding from chapters 2 and 3 now has a second, independent source that agrees with the first, which is worth more than it sounds: a number that two different tools produce from the same data is a number you can defend.

And all of it is now a notebook. When the April to June file arrives, change one path and run it. The checks run, the joins run, the shares update, and the question for the board becomes the interesting one: did anything change?

## Looking ahead

- **Worked exercise:** `chapter_05_guided.ipynb`. The chapter 2 worked exercise, in pandas: hour by pass type as a `pivot_table`, column shares, a two-line chart, and the same sentence about commuters and day-trippers.
- **On your own:** `chapter_05_solo.ipynb`. The chapter 3 exercises, in pandas: the checks, a `keep` mask, before and after, the five busiest end stations named by `merge`, round-trip share by region, and the four-sentence finding.
- **Before the next chapter:** look at every chart in chapters 1 to 5 and decide which were the right kind of chart for the question. Chapter 6 is about choosing.

## Appendix: reproducing this chapter {.unnumbered}

`chapter_05_lecture.ipynb` is generated from this page by `make notebooks_chapter_05`; the exercises come from `tools/build_chapter_05_exercises.py`. Every number quoted here is computed on the page from `data/raw/`, and every one of them appears somewhere in the chapter 2 or chapter 3 workbooks, which is the check.